# Per-category `min_ade` — KD arms on LCDrive val (n = 1000)

Every arm is scored **through the teacher's frozen action expert** (`StitchedAlpamayoR1`),
which is the head these arms were trained to drive. Scoring a student's own token head
instead gives a different — and for the CE-free arms, degenerate (`min_ade` 37.5239,
every generation malformed) — answer.

⚠️ Two KINDS of arm appear below. Most share the teacher's **untouched** expert, so they
measure cache quality. The `eos_*` arms **trained the expert** on the student's cache, and
the `prune_*` arms **removed expert layers** -- for those the decoder is no longer held
constant, so the teacher's 0.5776 is not a ceiling in the same sense.

`min_ade` is **best-of-6** samples against ground truth, so it is an *oracle* over modes.
Lower is better. Comparisons are **paired per clip**, never a difference of aggregates.

Colour encodes magnitude **within each row**, so each category is judged on its own scale:
a category where every arm does badly does not paint the whole row dark.

In [1]:
import json, csv, numpy as np, pandas as pd
from pathlib import Path

TRAIN = Path("/temp/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/training")
SCENARIOS = Path("/temp/achahe/physical_ai_av/lcdrive_physicalai_av_manifests"
                 "/lcdrive_val_primary_scenario_mysubset.csv")

# label -> stitched per-clip results file. Ordered ceiling -> floor.
ARMS = {
    # --- reference points -------------------------------------------------------------
    "teacher":        "stitch_4b_teacher.json",
    # --- cache distillation: the student's cache read by the teacher's UNTOUCHED expert
    "blockrandt_e1":  "stitch_4b_blockrandt_checkpoint-1598.json",
    "blockrandt_e2":  "stitch_4b_blockrandt_e3_checkpoint-3196.json",
    "blockrandt_e3":  "stitch_4b_blockrandt_e3_checkpoint-4794.json",
    "blockonly_e1":   "stitch_4b_blockonly_checkpoint-1598.json",
    "blockonly_e2":   "stitch_4b_blockonly_e3_checkpoint-3196.json",
    "blockonly_e3":   "stitch_4b_blockonly_e3_checkpoint-4794.json",
    "kvonly_e1":      "stitch_4b_kvonly.json",
    "kvonly_e3":      "stitch_4b_kvonly_e3_checkpoint-4794.json",
    "kvband_e1":      "stitch_4b_kvband_checkpoint-1598.json",
    "ce":             "stitch_4b_ce.json",
    # --- expert ADAPTED to the student's cache (blockrandt e3 VLM, frozen) -------------
    # ⚠️ These change the EXPERT, every arm above shares the teacher's untouched one. So the
    # teacher's 0.5776 is no longer a fixed ceiling in the same sense -- the decoder is no
    # longer held constant. What they measure is the deployable PAIR.
    "eos_step500":    "stitch_4b_eos_checkpoint-500.json",
    "eos_epoch1":     "stitch_4b_eos_checkpoint-1598.json",
    # --- teacher with 8 of 36 expert layers BYPASSED (see PRUNING.md) ------------------
    # ⚠️ NOT distillation arms: the teacher with layers removed, as an upper bound on any
    # pruned-expert student. A 28-layer expert distils toward THESE, not toward 0.5776.
    "prune_C":        "stitch_4b_teacher_pruneC.json",
    "prune_B":        "stitch_4b_teacher_pruneB.json",
    "prune_A":        "stitch_4b_teacher_pruneA.json",
}
ARMS = {k: v for k, v in ARMS.items() if (TRAIN / v).exists()}
print("arms found:", ", ".join(ARMS))

arms found: teacher, blockrandt_e1, blockrandt_e2, blockrandt_e3, blockonly_e1, blockonly_e2, blockonly_e3, kvonly_e1, kvonly_e3, kvband_e1, ce, eos_step500, prune_C, prune_B, prune_A


In [2]:
# Each stitched JSON is a flat list of per-clip records carrying `clip_id`, which is what
# makes the join to scenario labels -- and the paired tests -- possible at all.
scores = {
    arm: {r["clip_id"]: r["min_ade"] for r in json.loads((TRAIN / f).read_text())}
    for arm, f in ARMS.items()
}
category = {r["clip_uuid"]: r["scenario_category"]
            for r in csv.DictReader(SCENARIOS.open())}

# Restrict to clips every arm scored, so all columns describe the SAME clip set.
clips = [c for c in category if all(c in s for s in scores.values())]
df = pd.DataFrame({arm: [scores[arm][c] for c in clips] for arm in ARMS},
                  index=pd.Index([category[c] for c in clips], name="category"))
print(f"{len(clips)} clips x {len(ARMS)} arms")

table = df.groupby("category").mean()
table.insert(0, "n", df.groupby("category").size())
table = table.sort_values("n", ascending=False)

overall = df.mean().to_frame().T
overall.insert(0, "n", len(clips))
overall.index = ["ALL"]
table = pd.concat([table, overall])

print(table.round(3).to_string())   # plain-text fallback
table.round(3)

1000 clips x 15 arms
                                   n  teacher  blockrandt_e1  blockrandt_e2  blockrandt_e3  blockonly_e1  blockonly_e2  blockonly_e3  kvonly_e1  kvonly_e3  kvband_e1      ce  eos_step500  prune_C  prune_B  prune_A
General Training/Validation      341    0.385          1.412          1.173          1.171         1.511         1.294         1.296      1.871      1.670      2.644   8.957        0.996    0.598    0.716    0.759
Lane Keeping Curve                59    0.976          4.612          3.643          3.726         4.300         3.491         3.644      6.574      5.773      6.032   8.380        3.238    1.393    1.984    1.715
Lead Vehicle Following            56    0.625          1.819          1.435          1.419         2.001         1.637         1.475      2.567      2.483      3.446   9.384        1.417    0.747    0.830    0.854
Nudge Static Obstacle Maneuver    56    0.482          1.583          1.265          1.304         1.669         1.469     

,n,teacher,blockrandt_e1,blockrandt_e2,blockrandt_e3,blockonly_e1,blockonly_e2,blockonly_e3,kvonly_e1,kvonly_e3,kvband_e1,ce,eos_step500,prune_C,prune_B,prune_A
General Training/Validation,341,0.385,1.412,1.173,1.171,1.511,1.294,1.296,1.871,1.670,2.644,8.957,0.996,0.598,0.716,0.759
Lane Keeping Curve,59,0.976,4.612,3.643,3.726,4.300,3.491,3.644,6.574,5.773,6.032,8.380,3.238,1.393,1.984,1.715
Lead Vehicle Following,56,0.625,1.819,1.435,1.419,2.001,1.637,1.475,2.567,2.483,3.446,9.384,1.417,0.747,0.830,0.854
Nudge Static Obstacle Maneuver,56,0.482,1.583,1.265,1.304,1.669,1.469,1.372,1.911,1.725,2.314,4.529,1.039,0.635,0.685,0.683
Lane Keeping,53,0.554,1.743,1.482,1.406,1.837,1.493,1.403,2.343,2.086,2.761,4.030,1.176,0.689,0.818,0.848
Nudge Maneuver,53,0.602,1.604,1.426,1.387,1.775,1.479,1.417,1.974,1.812,2.207,2.963,1.392,0.684,0.793,0.763
Speed Control,53,0.597,3.220,2.907,2.628,3.297,2.805,2.667,3.302,3.160,4.171,6.575,2.390,0.776,0.996,0.875
Stop for Vehicle,48,0.447,1.380,1.264,1.059,1.490,1.294,1.218,2.980,2.822,3.058,3.120,0.911,0.553,0.682,0.635
Vulnerable Road Users (VRU),47,0.632,1.362,1.267,1.351,1.401,1.356,1.319,1.414,1.330,1.725,2.337,1.316,0.602,0.588,0.727
Lane Change,47,0.880,2.745,2.388,2.237,2.840,2.340,2.260,3.416,3.113,4.253,12.459,2.101,1.313,1.638,1.462


In [4]:
from matplotlib.colors import LinearSegmentedColormap

# Sequential = ONE hue, light -> dark (never a rainbow, never red/green: both fail CVD).
# Light = low min_ade = better.
SEQ = LinearSegmentedColormap.from_list("seq", ["#f2f7fb", "#c8dbeb", "#7fa9cd", "#3d6f9e", "#1b3d5c"])

arm_cols = [c for c in table.columns if c != "n"]

styled = (
    table.style
    # axis=1 -> normalise WITHIN each row, so every category is judged on its own scale.
    .background_gradient(cmap=SEQ, subset=arm_cols, axis=1, text_color_threshold=0.45)
    .format({"n": "{:.0f}", **{c: "{:.3f}" for c in arm_cols}})
    .set_caption("min_ade through the teacher's expert — lower is better; "
                 "colour is row-relative (per category)")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"),
                                          ("padding-bottom", "8px"), ("color", "#444")]},
        {"selector": "th", "props": [("font-weight", "600"), ("text-align", "right")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])
)
styled


,n,teacher,blockrandt_e1,blockrandt_e2,blockrandt_e3,blockonly_e1,blockonly_e2,blockonly_e3,kvonly_e1,kvonly_e3,kvband_e1,ce,eos_step500,prune_C,prune_B,prune_A
General Training/Validation,341,0.385,1.412,1.173,1.171,1.511,1.294,1.296,1.871,1.670,2.644,8.957,0.996,0.598,0.716,0.759
Lane Keeping Curve,59,0.976,4.612,3.643,3.726,4.300,3.491,3.644,6.574,5.773,6.032,8.380,3.238,1.393,1.984,1.715
Lead Vehicle Following,56,0.625,1.819,1.435,1.419,2.001,1.637,1.475,2.567,2.483,3.446,9.384,1.417,0.747,0.830,0.854
Nudge Static Obstacle Maneuver,56,0.482,1.583,1.265,1.304,1.669,1.469,1.372,1.911,1.725,2.314,4.529,1.039,0.635,0.685,0.683
Lane Keeping,53,0.554,1.743,1.482,1.406,1.837,1.493,1.403,2.343,2.086,2.761,4.030,1.176,0.689,0.818,0.848
Nudge Maneuver,53,0.602,1.604,1.426,1.387,1.775,1.479,1.417,1.974,1.812,2.207,2.963,1.392,0.684,0.793,0.763
Speed Control,53,0.597,3.220,2.907,2.628,3.297,2.805,2.667,3.302,3.160,4.171,6.575,2.390,0.776,0.996,0.875
Stop for Vehicle,48,0.447,1.380,1.264,1.059,1.490,1.294,1.218,2.980,2.822,3.058,3.120,0.911,0.553,0.682,0.635
Vulnerable Road Users (VRU),47,0.632,1.362,1.267,1.351,1.401,1.356,1.319,1.414,1.330,1.725,2.337,1.316,0.602,0.588,0.727
Lane Change,47,0.880,2.745,2.388,2.237,2.840,2.340,2.260,3.416,3.113,4.253,12.459,2.101,1.313,1.638,1.462


## Paired comparison: `blockonly` − `kvonly_e3`, within category

Negative favours `blockonly`. `z` is a paired one-sample statistic on the per-clip
differences, so it accounts for the fact that some categories are simply harder.

Colour here is **diverging** — two hues with a neutral midpoint at exactly 0 — because the
quantity has polarity (which arm wins), not magnitude.

In [4]:
# The comparison that matters now: adapting the EXPERT to the student's cache, versus
# the same student read by the teacher's untouched expert. Both use the blockrandt e3 VLM,
# so the only difference is whether the expert was allowed to move.
# Latest available expert-adapted arm (epoch-1 eval may still be running).
# df carries one column per arm found on disk, so key off that.
a = "eos_epoch1" if "eos_epoch1" in df.columns else "eos_step500"
b = "blockrandt_e3"
print(f"paired: {a} vs {b}")
rows = []
for cat, g in df.groupby("category"):
    d = (g[a] - g[b]).to_numpy()
    z = d.mean() / (d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 and d.std() > 0 else np.nan
    rows.append({"category": cat, "n": len(d), f"{a}": g[a].mean(),
                 f"{b}": g[b].mean(), "delta": d.mean(), "z": z})
d_all = (df[a] - df[b]).to_numpy()
rows.append({"category": "ALL", "n": len(d_all), f"{a}": df[a].mean(), f"{b}": df[b].mean(),
             "delta": d_all.mean(),
             "z": d_all.mean() / (d_all.std(ddof=1) / np.sqrt(len(d_all)))})
paired = pd.DataFrame(rows).set_index("category")

# Diverging: two hues + a NEUTRAL GREY midpoint, symmetric about 0 so the midpoint is
# genuinely "no difference" rather than wherever the data happens to centre.
DIV = LinearSegmentedColormap.from_list(
    "div", ["#1b3d5c", "#7fa9cd", "#eceff1", "#e0a367", "#8a4b12"])
lim = float(np.nanmax(np.abs(paired["delta"])))

styled_paired = (paired.style
   .background_gradient(cmap=DIV, subset=["delta"], vmin=-lim, vmax=lim, text_color_threshold=0.45)
   .format({"n": "{:.0f}", a: "{:.3f}", b: "{:.3f}", "delta": "{:+.3f}", "z": "{:+.2f}"})
   .set_caption(f"{a} − {b} per category · negative favours {a} · |z| > 2 ≈ significant")
   .set_table_styles([
       {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"),
                                         ("padding-bottom", "8px"), ("color", "#444")]},
       {"selector": "th.row_heading", "props": [("text-align", "left")]},
       {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
   ]))

print(paired.round(3).to_string())   # plain-text fallback
styled_paired

paired: eos_step500 vs blockrandt_e3
                                   n  eos_step500  blockrandt_e3  delta      z
category                                                                      
Cut-In                            36        1.282          1.317 -0.036 -0.154
General Training/Validation      341        0.996          1.171 -0.175 -4.146
Intersection Navigation           42        1.164          1.276 -0.112 -0.767
Lane Change                       47        2.101          2.237 -0.136 -1.163
Lane Keeping                      53        1.176          1.406 -0.230 -2.019
Lane Keeping Curve                59        3.238          3.726 -0.488 -1.821
Lead Vehicle Following            56        1.417          1.419 -0.003 -0.021
Merging                           44        1.578          1.576  0.002  0.016
Nudge Maneuver                    53        1.392          1.387  0.005  0.033
Nudge Static Obstacle Maneuver    56        1.039          1.304 -0.265 -2.040
Speed Control  

,n,eos_step500,blockrandt_e3,delta,z
category,,,,,
Cut-In,36,1.282,1.317,-0.036,-0.15
General Training/Validation,341,0.996,1.171,-0.175,-4.15
Intersection Navigation,42,1.164,1.276,-0.112,-0.77
Lane Change,47,2.101,2.237,-0.136,-1.16
Lane Keeping,53,1.176,1.406,-0.230,-2.02
Lane Keeping Curve,59,3.238,3.726,-0.488,-1.82
Lead Vehicle Following,56,1.417,1.419,-0.003,-0.02
Merging,44,1.578,1.576,+0.002,+0.02
Nudge Maneuver,53,1.392,1.387,+0.005,+0.03
